# Day 1 — Applying the cleaning decisions

Companion to `1.0-aroa-data-quality.ipynb`, which investigated the data and
recorded what should be done about each issue found. This notebook applies
those decisions and writes the analysis dataset to `data/processed/`.

Kept separate so the cleaning can be re-run on its own, without repeating the
investigation.

**Scope.** This produces a clean event log and a client-level table. It
deliberately stops short of journey reconstruction — collapsing repeated
`start` events, deciding what counts as one attempt, and defining completion
are day 3 decisions, made once the measurement methodology is agreed.

Every step reports what it removed, so the cost of each decision is visible.

## Setup

In [1]:
import pandas as pd
import numpy as np

from project_template.paths import RAW_DIR, PROCESSED_DIR
from project_template.config import CONFIG

pd.set_option("display.max_columns", 50)

FILES = CONFIG["files"]
FUNNEL = CONFIG["funnel"]

audit = []


def record(step, before, after, unit="events"):
    """Track what each cleaning step costs, so nothing disappears silently."""
    audit.append({
        "step": step,
        "before": before,
        "after": after,
        "removed": before - after,
        "% removed": round((before - after) / before * 100, 2) if before else 0,
        "unit": unit,
    })

## Load the raw data

In [2]:
demo = pd.read_csv(RAW_DIR / FILES["demo"])
roster = pd.read_csv(RAW_DIR / FILES["experiment"])
web = pd.concat(
    [pd.read_csv(RAW_DIR / f) for f in FILES["web"]],
    ignore_index=True,
)

print(f"demo    {len(demo):>9,} rows")
print(f"roster  {len(roster):>9,} rows")
print(f"web     {len(web):>9,} events")

demo       70,609 rows
roster     70,609 rows
web       755,405 events


## Step 1 — Parse the timestamps

`date_time` arrives as text. Nothing that depends on ordering or duration works
until it is a datetime.

In [3]:
web["date_time"] = pd.to_datetime(web["date_time"])

failed = web.date_time.isna().sum()
print(f"failed to parse: {failed}")
print(f"range: {web.date_time.min()} -> {web.date_time.max()}")

failed to parse: 0
range: 2017-03-15 00:03:03 -> 2017-06-20 23:59:57


## Step 2 — Drop exact duplicate events

Rows identical in client, visit, step and timestamp. The same interaction
cannot happen twice in the same second, so these are the tracker firing twice.

In [4]:
before = len(web)
web = web.drop_duplicates()
record("Drop duplicate events", before, len(web))

print(f"removed {before - len(web):,} duplicated events")

removed 10,764 duplicated events


## Step 3 — Drop the empty demographic rows

Fourteen to fifteen rows that are empty across almost every column. These are
empty records rather than unreported values.

In [5]:
before = len(demo)
demo = demo.dropna(subset=["clnt_age", "clnt_tenure_yr", "bal"], how="all")
record("Drop empty demo rows", before, len(demo), unit="clients")

print(f"removed {before - len(demo)} rows")
print(f"remaining missing values:\n{demo.isna().sum()[lambda s: s > 0].to_string() or '  none'}")

removed 14 rows
remaining missing values:
clnt_age    1


## Step 4 — Keep only the clients in the experiment

Two groups of clients cannot take part in the comparison:

- those with no `Variation`, who were never randomised
- those who appear in the web log but have no profile in `demo`

Both are excluded here and declared as limitations. This is the step that costs
the most, so it is worth seeing exactly what it removes.

In [6]:
assigned = roster.dropna(subset=["Variation"])
record("Clients: assigned to a group", len(roster), len(assigned), unit="clients")

# Only clients we can also profile
analysable = assigned[assigned.client_id.isin(demo.client_id)]
record("Clients: also profiled", len(assigned), len(analysable), unit="clients")

print(f"clients in roster           {len(roster):>8,}")
print(f"  assigned to a group       {len(assigned):>8,}")
print(f"  and with a profile        {len(analysable):>8,}")
print()
print(analysable.Variation.value_counts().to_string())

clients in roster             70,609
  assigned to a group         50,500
  and with a profile          50,488

Variation
Test       26961
Control    23527


In [7]:
before = len(web)
web = web[web.client_id.isin(analysable.client_id)]
record("Events from analysable clients", before, len(web))

print(f"events kept  {len(web):>9,}")
print(f"removed      {before - len(web):>9,}")
print(f"clients      {web.client_id.nunique():>9,}")
print(f"visits       {web.visit_id.nunique():>9,}")

events kept    317,135
removed        427,506
clients         50,488
visits          69,185


## Step 5 — Order the events and mark navigation

Mapping each step to its position in the funnel is what makes it possible to
tell forward progress from going back. These are descriptive flags, not KPIs:
whether a backward move counts as an error is decided on day 3.

In [8]:
STEP_ORDER = {step: i for i, step in enumerate(FUNNEL)}

web = web.sort_values(["visit_id", "date_time"]).reset_index(drop=True)
web["step_rank"] = web.process_step.map(STEP_ORDER)

prev = web.groupby("visit_id").step_rank.shift()
web["is_repeat"] = (web.step_rank == prev).fillna(False)
web["is_backward"] = (web.step_rank < prev).fillna(False)
web["steps_back"] = (prev - web.step_rank).where(web.is_backward, 0)

web[["client_id", "visit_id", "process_step", "date_time",
     "step_rank", "is_repeat", "is_backward"]].head(10)

,client_id,visit_id,process_step,date_time,step_rank,is_repeat,is_backward
0,3561384,100012776_37918976071_457913,confirm,2017-04-26 13:22:17,4,False,False
1,3561384,100012776_37918976071_457913,confirm,2017-04-26 13:23:09,4,True,False
2,7338123,100019538_17884295066_43909,start,2017-04-09 16:20:56,0,False,False
3,7338123,100019538_17884295066_43909,step_1,2017-04-09 16:21:12,1,False,False
4,7338123,100019538_17884295066_43909,step_2,2017-04-09 16:21:21,2,False,False
5,7338123,100019538_17884295066_43909,step_1,2017-04-09 16:21:35,1,False,True
6,7338123,100019538_17884295066_43909,step_1,2017-04-09 16:21:41,1,True,False
7,7338123,100019538_17884295066_43909,start,2017-04-09 16:21:45,0,False,True
8,7338123,100019538_17884295066_43909,start,2017-04-09 16:21:59,0,True,False
9,7338123,100019538_17884295066_43909,step_1,2017-04-09 16:22:04,1,False,False


## Step 6 — Build the analysis tables

Two outputs. The event log stays at event level, because journeys have not been
reconstructed yet. The client table joins the assignment to the demographics,
which is what the group comparison on day 2 needs.

In [9]:
events = web.merge(
    analysable[["client_id", "Variation"]], on="client_id", how="left"
)

print(f"events  {len(events):>9,} rows x {events.shape[1]} columns")
print(f"unassigned events after merge: {events.Variation.isna().sum()}")
events.head()

events    317,135 rows x 10 columns
unassigned events after merge: 0


,client_id,visitor_id,visit_id,process_step,date_time,step_rank,is_repeat,is_backward,steps_back,Variation
0,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:22:17,4,False,False,0.0,Test
1,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:23:09,4,True,False,0.0,Test
2,7338123,612065484_94198474375,100019538_17884295066_43909,start,2017-04-09 16:20:56,0,False,False,0.0,Test
3,7338123,612065484_94198474375,100019538_17884295066_43909,step_1,2017-04-09 16:21:12,1,False,False,0.0,Test
4,7338123,612065484_94198474375,100019538_17884295066_43909,step_2,2017-04-09 16:21:21,2,False,False,0.0,Test


In [10]:
clients = analysable.merge(demo, on="client_id", how="left")

print(f"clients {len(clients):>9,} rows x {clients.shape[1]} columns")
print()
print(clients.groupby("Variation").size().to_string())
clients.head()

clients    50,488 rows x 10 columns

Variation
Control    23527
Test       26961


,client_id,Variation,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth
0,9988021,Test,5.0,64.0,79.0,U,2.0,189023.86,1.0,4.0
1,8320017,Test,22.0,274.0,34.5,M,2.0,36001.90,5.0,8.0
2,4033851,Control,12.0,149.0,63.5,M,2.0,142642.26,5.0,8.0
3,1982004,Test,6.0,80.0,44.5,U,2.0,30231.76,1.0,4.0
4,9294070,Control,5.0,70.0,29.0,U,2.0,34254.54,0.0,3.0


## What the cleaning cost

Every step above, with what it removed. This is the audit trail behind the
decisions recorded in `1.0-aroa-data-quality.ipynb`.

In [11]:
pd.DataFrame(audit)

,step,before,after,removed,% removed,unit
0,Drop duplicate events,755405,744641,10764,1.42,events
1,Drop empty demo rows,70609,70595,14,0.02,clients
2,Clients: assigned to a group,70609,50500,20109,28.48,clients
3,Clients: also profiled,50500,50488,12,0.02,clients
4,Events from analysable clients,744641,317135,427506,57.41,events


## Save

In [12]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

events.to_parquet(PROCESSED_DIR / "events.parquet", index=False)
clients.to_parquet(PROCESSED_DIR / "clients.parquet", index=False)

for f in sorted(PROCESSED_DIR.glob("*.parquet")):
    print(f"{f.name:20} {f.stat().st_size / 1024**2:6.1f} MB")

clients.parquet         0.9 MB
events.parquet          6.9 MB


## Reading these files

From any notebook:

```python
import pandas as pd
from project_template.paths import PROCESSED_DIR

events = pd.read_parquet(PROCESSED_DIR / "events.parquet")
clients = pd.read_parquet(PROCESSED_DIR / "clients.parquet")
```

Parquet keeps the data types, so `date_time` comes back as a datetime and the
boolean flags stay boolean — no re-parsing, and much faster to load than CSV.

These files are not versioned in git. Re-run this notebook to rebuild them.